In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

ROOT         = Path("..")
DATA_MATCHED = ROOT / "data" / "matched"

In [3]:
pairs      = pd.read_csv(DATA_MATCHED / "pairs_lift.csv")
all_authors = pd.read_csv(DATA_MATCHED / "all_authors_within_paper.csv")

print("pairs_lift:            ", pairs.shape)
print("all_authors_within_paper:", all_authors.shape)
print("\nall_authors sample:")
print(all_authors[["author_id","conference","award_year","seniority"]].head(3))

pairs_lift:             (358, 9)
all_authors_within_paper: (2291, 10)

all_authors sample:
     author_id conference  award_year seniority
0  A5026521600       AAAI        2018    senior
1  A5014823249       AAAI        2018    junior
2  A5010677076       AAAI        2018    senior


In [4]:
# strip full URL to short ID (e.g. "A5026521600")
pairs["treated_short"] = pairs["treated_id"].str.extract(r"(A\d+)")
pairs["control_short"] = pairs["control_id"].str.extract(r"(A\d+)")

# for each author, collect all award years across all conferences
award_years_map = all_authors.groupby("author_id")["award_year"].apply(list).to_dict()

def has_prior_award(author_id, focal_year):
    return any(y < focal_year for y in award_years_map.get(author_id, []))

pairs["senior_has_prior"] = pairs.apply(
    lambda r: has_prior_award(r["control_short"], r["award_year"]), axis=1
)

print(f"Total pairs: {len(pairs)}")
print(f"Seniors with a prior award: {pairs['senior_has_prior'].sum()}")
print(f"Pairs remaining after filter: {(~pairs['senior_has_prior']).sum()}")

Total pairs: 358
Seniors with a prior award: 10
Pairs remaining after filter: 348


In [5]:
pairs_clean = pairs[~pairs["senior_has_prior"]].copy()

for metric, t_col, c_col in [
    ("Citations", "lift_cit_treated",   "lift_cit_control"),
    ("Works",     "lift_works_treated", "lift_works_control"),
]:
    stat, p = stats.wilcoxon(pairs_clean[t_col], pairs_clean[c_col])
    median_diff = (pairs_clean[t_col] - pairs_clean[c_col]).median()
    print(f"\n── Wilcoxon (filtered): {metric} ──")
    print(f"  Pairs: {len(pairs_clean)}")
    print(f"  Median lift diff (junior − senior): {median_diff:.3f}")
    print(f"  Statistic: {stat:.1f} | p-value: {p:.4f}")
    print(f"  Significant at 0.05: {'✓ YES' if p < 0.05 else '✗ NO'}")


── Wilcoxon (filtered): Citations ──
  Pairs: 348
  Median lift diff (junior − senior): -0.079
  Statistic: 28970.0 | p-value: 0.4583
  Significant at 0.05: ✗ NO

── Wilcoxon (filtered): Works ──
  Pairs: 348
  Median lift diff (junior − senior): -0.333
  Statistic: 23311.5 | p-value: 0.0002
  Significant at 0.05: ✓ YES
